# Low-N Protein Engineering: Round-0 → BO-EVO (Notebook)

Setup & Config

In [12]:
import os, sys, math, random, itertools, json, time
from dataclasses import dataclass
from typing import List, Tuple, Iterable, Optional, Dict

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel

from tqdm.auto import tqdm
SHOW_PROGRESS = True

In [13]:
RANDOM_SEED = 1337
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# User-configurable:
MODEL_FAMILY = "esm1v"   # "esm2" or "esm1v"
ESM_DEVICE   = None     # "cuda" / "cpu" / None (auto)
PCA_DIM      = 64
BATCH_SIZE   = 24       # proposed batch per round
XI           = 0.01     # EI exploration param
MIN_HD       = 2        # min Hamming distance between chosen proposals
MAX_MUT      = 2        # max mutation order around WT in search space
NB_BUDGET    = 30000    # cap on candidate count
AA_ALPHABET  = list("ACDEFGHIKLMNPQRSTVWY")

Data Classes & Utils

In [14]:
@dataclass
class LabeledSeq:
    seq: str
    fitness: float

@dataclass
class Proposal:
    seq: str
    mu: float
    sigma: float
    ei: float

def hamming(s1: str, s2: str) -> int:
    assert len(s1) == len(s2), "Sequences must have same length"
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))

def mutate_once(seq: str, positions: List[int]) -> Iterable[str]:
    for pos in positions:
        wt = seq[pos]
        for aa in AA_ALPHABET:
            if aa != wt:
                yield seq[:pos] + aa + seq[pos+1:]

def neighborhood(seq: str, positions: List[int], max_mutations: int = 2, budget: int = 50_000) -> List[str]:
    cand = set([seq])
    # single mutants
    for s in tqdm(mutate_once(seq, positions),
                  disable=not SHOW_PROGRESS, desc="Generating single mutants"):
        cand.add(s)

    # higher-order mutants (sampled)
    for k in range(2, max_mutations+1):
        combos = list(itertools.combinations(positions, k))
        for pos_tuple in tqdm(combos, disable=not SHOW_PROGRESS,
                              desc=f"Generating {k}-mutants"):
            choices = []
            for pos in pos_tuple:
                wt = seq[pos]
                choices.append([aa for aa in AA_ALPHABET if aa != wt])
            total = 1
            for c in choices:
                total *= len(c)
            samples = min(200, total)
            for _ in range(samples):
                aas = [random.choice(c) for c in choices]
                s = list(seq)
                for p, newaa in zip(pos_tuple, aas):
                    s[p] = newaa
                cand.add("".join(s))
            if len(cand) >= budget:
                break
        if len(cand) >= budget:
            break
    return list(cand)

Embedder

In [15]:
class Embedder:
    def __init__(self, family: str = "esm2", device: Optional[str] = None):
        self.family = family
        self.device = device
        self._mode = "fallback"
        self._embed_dim = 1280  # layer-33 hidden size for both 650M models
        self._rng = np.random.RandomState(RANDOM_SEED)
        self._setup()

    def _setup(self):
        try:
            import torch
            import esm
            self.torch = torch
            self.esm = esm
            if self.family == "esm2":
                self.model, self.alphabet = esm.pretrained.esm2_t33_650M_UR50D()
            elif self.family == "esm1v":
                self.model, self.alphabet = esm.pretrained.esm1v_t33_650M_UR90S_1()
            else:
                raise ValueError("family must be 'esm2' or 'esm1v'")
            if self.device is None:
                self.device = "cuda" if torch.cuda.is_available() else "cpu"
            self.model = self.model.eval().to(self.device)
            self.batch_converter = self.alphabet.get_batch_converter()
            self._mode = "esm"
        except Exception:
            # Fallback: one-hot → random projection → mean pooling
            self._mode = "fallback"
            self._proj = self._rng.normal(0, 1/np.sqrt(20), size=(20, self._embed_dim))

    @property
    def dim(self) -> int:
        return self._embed_dim

    def _one_hot_embed(self, seq: str) -> np.ndarray:
        mat = np.zeros((len(seq), 20), dtype=np.float32)
        aa_to_idx = {aa:i for i,aa in enumerate(AA_ALPHABET)}
        for i, aa in enumerate(seq):
            if aa in aa_to_idx:
                mat[i, aa_to_idx[aa]] = 1.0
        return (mat @ self._proj).mean(axis=0)

    def encode(self, seqs: List[str], batch_size: int = 16) -> np.ndarray:
        if self._mode != "esm":
            it = tqdm(seqs, disable=not SHOW_PROGRESS, desc="Embedding (fallback)")
            return np.vstack([self._one_hot_embed(s) for s in it])

        embs = []
        total = math.ceil(len(seqs) / batch_size)
        with self.torch.no_grad():
            for start in tqdm(range(0, len(seqs), batch_size),
                              total=total, disable=not SHOW_PROGRESS,
                              desc=f"Embedding ({self.family}) batches"):
                batch = [("seq", s) for s in seqs[start:start+batch_size]]
                _, _, toks = self.batch_converter(batch)
                toks = toks.to(self.device)
                out = self.model(toks, repr_layers=[33], return_contacts=False)
                reps = out["representations"][33]                 # [B, L, 1280]
                pooled = reps[:, 1:-1, :].mean(dim=1).cpu().numpy()  # mean over residues
                embs.append(pooled)
        return np.vstack(embs)

Surrogate

In [16]:
class GPSurrogate:
    def __init__(self, pca_dim: int = 64):
        self.pca_dim = pca_dim
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=pca_dim, random_state=RANDOM_SEED)
        kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=5.0, length_scale_bounds=(1e-2, 1e3)) \
                 + WhiteKernel(noise_level=1e-4, noise_level_bounds=(1e-8, 1e-1))
        self.gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha=0.0,
            normalize_y=True,
            random_state=RANDOM_SEED,
            n_restarts_optimizer=2
        )
        self.fitted = False

    def fit(self, X_emb: np.ndarray, y: np.ndarray):
        Z = self.scaler.fit_transform(X_emb)
        Z = self.pca.fit_transform(Z)
        self.gp.fit(Z, y)
        self.fitted = True

    def predict(self, X_emb: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        assert self.fitted
        Z = self.scaler.transform(X_emb)
        Z = self.pca.transform(Z)
        mu, sigma = self.gp.predict(Z, return_std=True)
        return mu, sigma

Expected Improvement

In [17]:
from scipy.stats import norm

def expected_improvement(mu: np.ndarray, sigma: np.ndarray, best_y: float, xi: float = 0.01) -> np.ndarray:
    imp = mu - best_y - xi
    Z = np.zeros_like(mu)
    nonzero = sigma > 0
    Z[nonzero] = imp[nonzero] / sigma[nonzero]
    ei = np.zeros_like(mu)
    ei[nonzero] = imp[nonzero] * norm.cdf(Z[nonzero]) + sigma[nonzero] * norm.pdf(Z[nonzero])
    return np.maximum(ei, 0.0)

Diversity Picker

In [18]:
def pick_diverse(proposals: List[Proposal], k: int, min_hd: int) -> List[Proposal]:
    chosen: List[Proposal] = []
    for p in sorted(proposals, key=lambda x: x.ei, reverse=True):
        if len(chosen) >= k:
            break
        if all(hamming(p.seq, c.seq) >= min_hd for c in chosen) or len(chosen) == 0:
            chosen.append(p)
    return chosen[:k]

BO-EVO

In [19]:
class BOEVO:
    def __init__(self, embedder: Embedder, surrogate: GPSurrogate, wt_seq: str,
                 mut_positions: List[int], max_mut: int = 2, neighborhood_budget: int = 50_000):
        self.embedder = embedder
        self.surrogate = surrogate
        self.wt_seq = wt_seq
        self.positions = mut_positions
        self.max_mut = max_mut
        self.nb_budget = neighborhood_budget

    def propose(self, labeled: List[LabeledSeq], batch_size: int = 24, xi: float = 0.01,
                min_hd_between: int = 2) -> List[Proposal]:
        # Train surrogate on Round-0 data
        X_train = self.embedder.encode(
            [x.seq for x in tqdm(labeled, disable=not SHOW_PROGRESS, desc="Embedding labeled")]
        )
        y_train = np.array([x.fitness for x in labeled], dtype=float)
        self.surrogate.fit(X_train, y_train)

        # Build neighborhood (candidate pool)
        cand = neighborhood(self.wt_seq, self.positions, self.max_mut, self.nb_budget)
        tested = set(x.seq for x in labeled)
        cand = [s for s in cand if s not in tested]

        # Score candidates
        X_cand = self.embedder.encode(cand)  # already has tqdm
        mu, sigma = self.surrogate.predict(X_cand)

        best_y = float(np.max(y_train))
        ei = expected_improvement(mu, sigma, best_y, xi=xi)

        proposals = []
        for c, m, s, e in tqdm(zip(cand, mu, sigma, ei),
                               total=len(cand), disable=not SHOW_PROGRESS,
                               desc="Packaging proposals"):
            proposals.append(Proposal(seq=c, mu=float(m), sigma=float(s), ei=float(e)))

        # Diversity selection
        chosen = pick_diverse(
            sorted(proposals, key=lambda x: x.ei, reverse=True),
            k=batch_size, min_hd=min_hd_between
        )
        return chosen

Load Round 0

In [20]:
ref_df = pd.read_csv("../data/ref_gfp.csv")
WT = ref_df.iloc[0]["target_seq"].strip().upper()
print("WT length:", len(WT))
print(WT[:60] + ("..." if len(WT) > 60 else ""))

WT length: 238
MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTL...


In [21]:
dms_path = "../data/GFP_AEQVI_Sarkisyan_2016.csv"
dms = pd.read_csv(dms_path)

seq_col = "mutated_sequence"
fit_col = "DMS_score"

# Clean and filter
dms[seq_col] = dms[seq_col].astype(str).str.strip().str.upper()
# Keep only sequences that match WT length and are valid AAs
valid_aas = set("ACDEFGHIKLMNPQRSTVWY")
def is_valid_seq(s): 
    return (len(s) == len(WT)) and all(ch in valid_aas for ch in s)

dms = dms[dms[seq_col].map(is_valid_seq)].copy()

# Drop missing fitness
dms = dms[~dms[fit_col].isna()].copy()

# z-score
dms["fitness_norm"] = (dms[fit_col] - dms[fit_col].mean()) / (dms[fit_col].std(ddof=0) + 1e-8)

print("DMS rows after cleaning:", len(dms))
dms.head()

DMS rows after cleaning: 51714


,mutant,mutated_sequence,DMS_score,DMS_score_bin,fitness_norm
0,K3R:V55A:Q94R:A110T:D117G:M153K:D216A,MSRGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,1.301030,0,-1.281845
1,K3Q:V16A:I167T:L195Q,MSQGEELFTGVVPILAELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,3.137350,1,0.452883
2,K3Q:Y143C:N164D:S205P:A227T,MSQGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,1.553913,0,-1.042952
3,K3Q:Y143N:V193A,MSQGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,3.404237,1,0.705005
4,K3R,MSRGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,3.738586,1,1.020856


In [22]:
use_col = "fitness_norm" if "fitness_norm" in dms.columns else fit_col

# Convert to the pipeline’s LabeledSeq list
labeled_data: List[LabeledSeq] = [
    LabeledSeq(seq=row[seq_col], fitness=float(row[use_col]))
    for _, row in dms.iterrows()
]

# Ensure WT is included with a neutral baseline
if WT not in set(dms[seq_col].values):
    labeled_data.append(LabeledSeq(seq=WT, fitness=float(np.median(dms[use_col]))))

print("Round-0 labeled count:", len(labeled_data))

# Derive mutable positions from DMS diversity: positions where at least one variant != WT
def diff_positions(wt: str, seqs: List[str]) -> List[int]:
    L = len(wt)
    vary = []
    for i in range(L):
        for s in seqs:
            if s[i] != wt[i]:
                vary.append(i)
                break
    return vary

mutate_positions = diff_positions(WT, dms[seq_col].values.tolist())
print("Mutable positions inferred from DMS:", len(mutate_positions))
print(mutate_positions[:50])

Round-0 labeled count: 51715
Mutable positions inferred from DMS: 233
[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]


In [23]:
LOW_N = None  # e.g., 48

if LOW_N is not None and LOW_N < len(labeled_data):
    rng = np.random.default_rng(1337)
    pool = rng.choice(labeled_data, size=min(5000, len(labeled_data)), replace=False).tolist()

    chosen = [pool[0]]
    for _ in tqdm(range(1, LOW_N), disable=not SHOW_PROGRESS, desc=f"Subsampling to LOW_N={LOW_N}"):
        best = None
        best_min_hd = -1
        for cand in pool:
            dmin = min(hamming(cand.seq, c.seq) for c in chosen)
            if dmin > best_min_hd:
                best_min_hd = dmin
                best = cand
        chosen.append(best)
    labeled_data = chosen
    print(f"Subsampled to LOW_N={len(labeled_data)} diverse sequences.")

Init Models

In [24]:
embedder = Embedder(family=MODEL_FAMILY, device=ESM_DEVICE)
surrogate = GPSurrogate(pca_dim=PCA_DIM)
bo = BOEVO(embedder, surrogate, wt_seq=WT,
           mut_positions=mutate_positions, max_mut=MAX_MUT,
           neighborhood_budget=NB_BUDGET)

print(f"Embedder mode: {embedder.__dict__.get('_mode')}, family={MODEL_FAMILY}, dim={embedder.dim}")

C:\Users\timap\AppData\Roaming\Python\Python312\site-packages\esm\pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(


Embedder mode: esm, family=esm1v, dim=1280


Propose Round 1

In [25]:
proposals = bo.propose(
    labeled=labeled_data,
    batch_size=BATCH_SIZE,
    xi=XI,
    min_hd_between=MIN_HD
)

len(proposals), proposals[0] if proposals else None

Embedding labeled:   0%|          | 0/51715 [00:00<?, ?it/s]

Embedding (esm1v) batches:   0%|          | 0/3233 [00:00<?, ?it/s]

KeyboardInterrupt: 

Inspect proposals

In [ ]:
props_df = pd.DataFrame([{
    "seq": p.seq, "mu": p.mu, "sigma": p.sigma, "ei": p.ei,
    "hd_to_WT": hamming(p.seq, WT)
} for p in sorted(proposals, key=lambda x: x.ei, reverse=True)])

display(props_df.head(20))

# Save if you want to send to synthesis / wet lab
# props_df.to_csv("round1_proposals.csv", index=False)

Plot round 1

In [ ]:
import matplotlib.pyplot as plt

# EI distribution
plt.figure()
plt.hist(props_df["ei"].values, bins=30)
plt.title("EI distribution (Round 1)")
plt.xlabel("EI")
plt.ylabel("Count")
plt.show()

# mu vs sigma
plt.figure()
plt.scatter(props_df["mu"].values, props_df["sigma"].values, s=10)
plt.title("Predictive mean vs std (Round 1)")
plt.xlabel("mu")
plt.ylabel("sigma")
plt.show()